In [ ]:
#using Pkg
#Pkg.activate("../../../")
#Pkg.instantiate()

In [2]:
using LAlatex, Latexify, LaTeXStrings
function p(s) L_interp(L"\text{$(s)}\quad \rightarrow \;\;", Dict("s" => s)) end
;

<div style="float:center;width:100%;text-align:center;">
<strong style="height:100px;color:darkred;font-size:40px;">HTML and LaTeX Rendering Utilities for Julia</strong>
</div>

# 1. HTML Functions

Use these helpers to emit styled HTML in notebooks.
* `show_html` returns a small HTML wrapper and `pr` is a paragraph-friendly alias.
* `show_side_by_side` stacks multiple blocks without writing raw HTML each time.


This notebook covers

- Common `l_show` patterns for matrices and tuples.
- Backend-aware symbol creation and display.
- Formatting helpers for row-echelon and highlights.


<div style="float:left;padding-left:1cm;">

| **Function**         | **Arguments** | **Description** |
|----------------------|----------------|----------------|
| *show_html*|( txt;sz=20,color="darkred",justify="left",height=15,width=100,env="strong")| display HTML text line |
| *show_html*|( txt1, txt2;sz1=20,sz2=20,color="darkred",justify="left",width=100,height=15, env="strong")| display 2 HTML text lines |
| *pr*|( txt;sz=15,color="black",justify="left",height=15,width=100,env="p")| print HTML text|
| *capture_output*|( f, args...)| support function to capture print output |
| *show_side_by_side*|( captured_outputs, titles=nothing)| print captured outputs in side by side <div>|
</div>

In [3]:
display(show_html("A title"))
display(pr(L"\qquad"*"we can print formulae "*L"\sqrt{1+x^2}", color="olive"))
show_html("Title", "with Subtitle", color="blue", sz1=24, sz2=12, justify="center")
#println()

LAlatex.HTMLOut("<div style=\"font-size: 20px; color: darkred; text-align: left; height: 15px; width: 100%;\">\n  <strong>A title</strong>\n</div>\n")

LAlatex.HTMLOut("<div style=\"font-size: 15px; color: olive; text-align: left; height: 15px; width: 100%;\">\n  <p>\$\\qquad\$we can print formulae \$\\sqrt{1+x^2}\$</p>\n</div>\n")

LAlatex.HTMLOut("<div style=\"color: blue; text-align: center; min-height: 15px; width: 100%;\">\n  <div style=\"font-size: 24px;\"><strong>Title</strong></div>\n  <div style=\"font-size: 12px;\"><strong>with Subtitle</strong></div>\n</div>\n")

In [4]:
# Example Use of display_side_by_side
function func1(x)
    println("Output of func1 with argument $x")
end

function func2(y)
    println("Output of func2 with argument $y")
    println("Another line from func2")
end

function func3(z)
    println("Output of func3 with argument $z")
    println("Second line from func3")
    println("Third line from func3")
end

# Capture outputs
outputs = [
    capture_output(func1, 1),
    capture_output(func2, 2),
    capture_output(func3, 3)
]

# Display side by side with custom titles
show_side_by_side(outputs, ["Function 1", "Function 2", "Function 3"])

LAlatex.SideBySideHTML("<div style=\"display: flex; justify-content: space-between;\">\n<div style=\"flex: 1; align-content:flex-start; margin-right: 10px;\">\n<h4>Function 1</h4>\n<pre>Output of func1 with argument 1\n</pre>\n</div>\n<div style=\"flex: 1; align-content:flex-start; margin-right: 10px;\">\n<h4>Function 2</h4>\n<pre>Output of func2 with argument 2\nAnother line from func2\n</pre>\n</div>\n<div style=\"flex: 1; align-content:flex-start; margin-right: 10px;\">\n<h4>Function 3</h4>\n<pre>Output of func3 with argument 3\nSecond line from func3\nThird line from func3\n</pre>\n</div>\n</div>")

# 2. LaTeX Rendering Function

This section shows the `L_show`/`l_show`/`py_show` family. `L_show` returns raw LaTeX for copy/paste, `l_show` returns a `LaTeXString` for Julia notebooks, and `py_show` sends LaTeX to a Python notebook.


The **l_show** functions allow easy display of julia scalars, vectors and matrices mixed with strings.
* *L_show*  produces a LaTeX representation of the arguments which could be copied into LaTeX documents
* *l_show*  uses *L_show* internally to render the resulting expressions in the julia notebook
* *py_show* uses *L_show* internally to render the resulting expressions in a python notebook

**Function Signature**
```julia
L_show(objs...; arraystyle=:parray, color=nothing, number_formatter=nothing,
       inline=true, factor_out=true, per_element_style=nothing)
```
**Arguments**
<div style="float:left;padding-left:1cm;">

| **Argument**         | **Type**                     | **Description** |
|----------------------|----------------------------|----------------|
| `objs...`           | Any                         | Numbers, matrices, vectors, `BlockArray`, `SymPy` expressions, etc. |
| `arraystyle`        | `Symbol`                    | LaTeX matrix format (`:bmatrix`, `:pmatrix`, `:array`, etc.). |
| `setstyle`          | `Symbol`                    | LaTeX set delimiter format (`:bmatrix`, `:pmatrix`, `:array`, etc.). |
| `color`             | `Union{Nothing, String}`    | Text color using `\textcolor{}` in LaTeX. |
| `number_formatter`  | `Union{Nothing, Function}`  | Function to format numbers before LaTeX conversion. |
| `factor_out`        | `Bool`                      | Factor out common denominators in rational entries. |
| `per_element_style` | `Union{Nothing, Function}`  | Function `(x, i, j, formatted) -> styled_string` for per-element formatting. |
| `separator`         | `Union{String,LaTeXString}` | Separator string to be inserted between tuple objects. |
| `symopts`           | `named list`                | symbolic transforms (simplify, expand, collect, factor) |
| `inline`            | `Bool`                      | If `true`, returns `$ ... $`; otherwise, `\[ ... \]`. |
</div>

**Supported Objects**
<div style="float:left;padding-left:1cm;">
<table>
  <tr>
    <th>Object Type</th>
    <th>Description & Example</th>
  </tr>
  <tr>
    <td><code>String</code> / <code>LaTeXString</code></td>
    <td>
      Arbitrary text or inline LaTeX code.<br>
      <code>L_show("Hello, world")</code>
    </td>
  </tr>
  <tr>
    <td><code>Number</code></td>
    <td>
      Integers, floats, rationals, complex numbers; displayed as scalars.<br>
      <code>L_show(3//4)</code>
    </td>
  </tr>
  <tr>
    <td><code>Vector</code> / <code>Matrix</code></td>
    <td>
      Julia arrays of any numeric or symbolic type, rendered with the selected <code>arraystyle</code>.<br>
      <code>L_show([1,2,3])</code>
    </td>
  </tr>
  <tr>
    <td><code>BlockArray</code></td>
    <td>
      Block-structured arrays are supported and displayed with correct partitioning.<br>
      <code>L_show(BlockArray([1 2; 3 4], [1,1], [1,1]))</code>
    </td>
  </tr>
  <tr>
    <td><code>Set</code></td>
    <td>
      Mathematical sets displayed with configurable <code>setstyle</code> delimiters.<br>
      <code>L_show(Set([1,2,3]))</code>
    </td>
  </tr>
  <tr>
    <td><code>Tuple</code></td>
    <td>
      Collections of heterogeneous objects, joined with <code>separator</code>.<br>
      <code>L_show((1,"a",[2,3]))</code>
    </td>
  </tr>
  <tr>
    <td><code>NamedTuple</code></td>
    <td>
      Key–value pairs with symbolic names, displayed as structured tuples.<br>
      <code>L_show((a=1, b=[2,3]))</code>
    </td>
  </tr>
  <tr>
    <td><code>SymPy</code> expressions</td>
    <td>
      Symbolic terms from <code>SymPy.jl</code>, including polynomials, rational functions, etc.<br>
      <code>L_show(sympy.symbols("x")^2 + 2)</code>
    </td>
  </tr>
  <tr>
    <td>Linear combinations</td>
    <td>
      Results from <code>lc(...)</code>, displayed as sums of scalar–vector products with proper LaTeX formatting.<br>
      <code>L_show(lc([2, -1], [[1,0],[0,1]]))</code>
    </td>
  </tr>
  <tr>
    <td>Nested structures</td>
    <td>
      Combinations of the above (e.g. tuple of matrices, set of vectors).<br>
      <code>L_show((Set([1,2]), [3,4]))</code>
    </td>
  </tr>
</table>
</div>

**Support Functions**
<div style="float:left;padding-left:1cm;">
    
| **Function**         | **Argument**               |
|----------------------|----------------------------|
| *bold_formatter* |(x, i, j, formatted_x)|
| *italic_formatter* |(x, i, j, formatted_x)|
| *color_formatter* |(x, i, j, formatted_x; color="red")|
| *conditional_color_formatter* |(x, i, j, formatted_x)|
| *highlight_large_values* |(x, i, j, formatted_x; threshold=10)|
| *underline_formatter* |(x, i, j, formatted_x1)|
| *overline_formatter* |(x, i, j, formatted_x)|
| *scientific_formatter* |(x; digits=2)|
| *percentage_formatter* |(x; digits=2)|
| *exponential_formatter* |(x; digits=2)|
| *combine_formatters* |(formatters, x, i, j, formatted_x)|
| *tril_formatter* | (x, i, j, formatted_x; k = 0, color = "red", c1 = 1, c2 = typemax(Int)) |
| *diagonal_blocks_formatter* | (x, i, j, formatted_x; blocks::Vector{Int}, colors ["red"]) |
| *block_formatter* | (x, i, j, formatted_x; r1 = 1, r2 = 1, c1 = 1, c2 = 1, color = "red") |
| *rowechelon_formatter* | (x, i, j, formatted_x; pivots::Vector{Int}, color="red") |
</div>

## 2.1. Scalars

Numbers, rationals, and symbols render as inline LaTeX. Use `color=` or formatter helpers when you want highlighting without changing the underlying value.


In [5]:
# Integers and Floats
display(l_show(p("The number   42"),   42))                 # Expected: 42
display(l_show(p("The number 3.14"), 3.14))                 # Expected: 3.14

L"$\text{The number   42}\quad \rightarrow \;\; 42$
"

L"$\text{The number 3.14}\quad \rightarrow \;\; 3.14$
"

In [6]:
# Rational Numbers
display(l_show(p("The fraction 1//3"), 1//3))               # Expected: \frac{1}{3}

# Complex Numbers
display(l_show(p("The complex number 2+3i"), 2 + 3im))      # Expected: 2 + 3i
display(l_show(p("The complex number -i"), -im//1))            # Expected: -i

L"$\text{The fraction 1//3}\quad \rightarrow \;\; \frac{1}{3}$
"

L"$\text{The complex number 2+3i}\quad \rightarrow \;\; 2+3\mathit{i}$
"

L"$\text{The complex number -i}\quad \rightarrow \;\; -\mathit{i}$
"

In [7]:
# With Color and Formatting
display(l_show((text="1//3 colored red: ", color="black"), 1//3, color="red")) # Expected: \frac{1}{3}  colored red
display(l_show("42 bold -> ", 42, number_formatter=x -> "\\textbf{$x}"))       # Expected: bold 42

L"$\textcolor{black}{\text{1//3 colored red: }} \textcolor{red}{\frac{1}{3}}$
"

L"$\text{42 bold -> } \textbf{42}$
"

## 2.2 Symbols

This example shows how to create symbols with both backends. Switch the backend, then call `syms` to get Symbolics or SymPy symbols.


In [8]:
# Backend switching: Symbolics vs SymPy
set_backend!(:sympy)
x_py, y_py = syms(:x, :y)
display(l_show(x_py,", ", y_py))

set_backend!(:symbolics)
x_sym, y_sym = syms(:x, :y)
display(l_show(x_sym,", ", y_sym))


L"$x , y$
"

L"$x  , y $
"

In [9]:
# Symbols and SymPy Symbols
display(l_show(p("The symbol :x" ), :x))                             # Expected: x

L"$\text{The symbol :x}\quad \rightarrow \;\; x $
"

### Symbolic display controls

Use `symopts` to apply optional transforms at display time.

In [10]:
set_backend!(:symbolics)
x, y = syms(:x, :y)
expr = (x + y)^2
display(l_show(expr; symopts=(expand=true,)))

L"$x^{2} + 2 ~ x ~ y + y^{2} $
"

### Mixed matrices

Use `mixed_matrix` or `@mixed_matrix` when combining Symbolics/SymPy symbols with numerical values.  
The macro keeps the `[ ... ]` syntax but avoids Julia promotion errors.


In [11]:
# Example: Symbolics mixed matrix
set_backend!(:symbolics)
using Symbolics
x, y = syms(:x, :y)
F = @mixed_matrix [1//2 x; (1 + im)//3 2*y]
display(l_show("factored ", "F = ", F, L",\qquad F^2 = ", F^2))
l_show("actual   ", "F = ", F, L",\qquad F^2 = ", F^2, factor_out=false)

L"$\text{factored } \text{F = } \frac{1}{6} \left(\begin{array}{rr}
3 & 6 ~ x \\
2+2\mathit{i} & 12 ~ y \\
\end{array}\right) ,\qquad F^2 = \frac{1}{12} \left(\begin{array}{rr}
3 + 4 ~ x+4 ~ x\mathit{i} & 6 ~ x + 24 ~ x ~ y \\
2 + 8 ~ y+2 + 8 ~ y\mathit{i} & 4 ~ x + 48 ~ y^{2}+4 ~ x\mathit{i} \\
\end{array}\right)$
"

L"$\text{actual   } \text{F = } \left(\begin{array}{rr}
\frac{1}{2} & x \\
\frac{1}{3}+\frac{1}{3}\mathit{i} & 2 ~ y \\
\end{array}\right) ,\qquad F^2 = \left(\begin{array}{rr}
\frac{1}{4} + \frac{1}{3} ~ x+\frac{1}{3} ~ x\mathit{i} & \frac{1}{2} ~ x + 2 ~ x ~ y \\
\frac{1}{6} + \frac{2}{3} ~ y+\frac{1}{6} + \frac{2}{3} ~ y\mathit{i} & \frac{1}{3} ~ x + 4 ~ y^{2}+\frac{1}{3} ~ x\mathit{i} \\
\end{array}\right)$
"

## 2.3. Strings and LaTeXStrings

Plain strings are escaped and wrapped in `\text{...}`. Use `LaTeXString` or `L"..."` when you want raw LaTeX to pass through.


In [12]:
display(l_show("Hello, World!"))                # Expected: \text{Hello, World!}
display(l_show("text_with_underscores"))        # Expected: \text{text\_with\_underscores}
display(l_show(L"A = B + C"))                   # Expected: A = B + C (keeps LaTeX raw)
display(l_show(L"A = B \cdot C", color="red"))  # Expected: \textcolor{red}{A = B \cdot C}

L"$\text{Hello, World!}$
"

L"$\text{text\_with\_underscores}$
"

L"$A = B + C$
"

L"$\textcolor{red}{A = B \cdot C}$
"

## 2.4. Tuples and Sets

Tuples concatenate multiple items; `set(...)` groups entries with delimiters  
and lets you control `separator`, `setstyle`, and `color` locally.


In [13]:
t1 = (1.234, L"\alpha", 3.14159)
display(l_show("Simple case: ", t1))
t2 = (π, :beta, :my_var, 2.71, 2//3, 4im//5)
display(l_show("Change the separator: ", t2, separator="; "))
t3 = (1.234, L"\alpha", (5, "nested"))
display(l_show("Nested list: ", t3))

L"$\text{Simple case: } 1.234,\alpha,3.14159$
"

L"$\text{Change the separator: } π;beta ;my_var ;2.71;\frac{2}{3};\frac{4}{5}\mathit{i}$
"

L"$\text{Nested list: } 1.234,\alpha,5,\text{nested}$
"

In [14]:
t4 = (m=[1 2; 3 4], arraystyle=:bmatrix, color="green")
display(l_show(p("tuple with named tuple formatting "), t4))

t5 = (1 + 2im, 3 - 4im)
display(l_show(p("tuple with complex numbers: "), t5))

A = [1//2 1//3; 3//4 2//5]
B = [5 6; 7 8im]

display(l_show( set("tuple of arrays: ", (A, arraystyle=:barray, color="green"), B, color="red", separator=L",\quad")))


t6 = (A, B, transpose(A), 2//3*B')
display(l_show("Tuple with text and arrays: ", t6, setstyle=:array, arraystyle=:parray, separator=L",\quad"))

L"$\text{tuple with named tuple formatting }\quad \rightarrow \;\; \textcolor{green}{\begin{bmatrix}
1 & 2 \\
3 & 4 \\
\end{bmatrix}}$
"

L"$\text{tuple with complex numbers: }\quad \rightarrow \;\; 1+2\mathit{i},3-4\mathit{i}$
"

L"$\textcolor{red}{\left\{ \textcolor{red}{\text{tuple of arrays: }} ,\quad \textcolor{green}{\frac{1}{60} \left[\begin{array}{rr}
30 & 20 \\
45 & 24 \\
\end{array}\right]} ,\quad \textcolor{red}{\left(\begin{array}{rr}
5 & 6 \\
7 & 8\mathit{i} \\
\end{array}\right)} \right\}}$
"

L"$\text{Tuple with text and arrays: } \frac{1}{60} \left(\begin{array}{rr}
30 & 20 \\
45 & 24 \\
\end{array}\right),\quad\left(\begin{array}{rr}
5 & 6 \\
7 & 8\mathit{i} \\
\end{array}\right),\quad\frac{1}{60} \left(\begin{array}{rr}
30 & 45 \\
20 & 24 \\
\end{array}\right),\quad\frac{1}{3} \left(\begin{array}{rr}
10 & 14 \\
12 & -16\mathit{i} \\
\end{array}\right)$
"

In [15]:
t7 = ((matrix=[1 2; 3 4], arraystyle=:bmatrix, color="blue"),
      42, " text", L"|\quad|", π, :gamma)

display(l_show("Mixed Elements Tuple: ", t7))

L"$\text{Mixed Elements Tuple: } \textcolor{blue}{\begin{bmatrix}
1 & 2 \\
3 & 4 \\
\end{bmatrix}},42,\text{ text},|\quad|,π,gamma $
"

In [16]:
v1 = [1, 2, 3]  # Column Vector
v2 = [4 5 6]    # Row Vector

t8 = (v1, v2)
display(l_show("Tuple of Row and Column vectors: ", t8, arraystyle=:bmatrix))
t8a = (v1', v2')
display(l_show("Tuple of Column and Row vectors: ", t8a, arraystyle=:bmatrix))
t8b = (transpose(v1), transpose(v2))
bold_green_formatter = (x,i,j,fx) -> combine_formatters(
    [bold_formatter,
     (x,i,j,fx) -> color_formatter(x,i,j,fx; color="green")],
    x,i,j,fx
)
display(l_show("Tuple of Column and Row vectors: ", t8a, arraystyle=:pmatrix,
                per_element_style=bold_green_formatter))

L"$\text{Tuple of Row and Column vectors: } \begin{bmatrix}
1 \\
2 \\
3 \\
\end{bmatrix},\begin{bmatrix}
4 & 5 & 6 \\
\end{bmatrix}$
"

L"$\text{Tuple of Column and Row vectors: } \begin{bmatrix}
1 & 2 & 3 \\
\end{bmatrix},\begin{bmatrix}
4 \\
5 \\
6 \\
\end{bmatrix}$
"

L"$\text{Tuple of Column and Row vectors: } \begin{pmatrix}
\textcolor{green}{\boldsymbol{1}} & \textcolor{green}{\boldsymbol{2}} & \textcolor{green}{\boldsymbol{3}} \\
\end{pmatrix},\begin{pmatrix}
\textcolor{green}{\boldsymbol{4}} \\
\textcolor{green}{\boldsymbol{5}} \\
\textcolor{green}{\boldsymbol{6}} \\
\end{pmatrix}$
"

In [17]:
t9 = (1//2, 3//4, 5//6)
display(l_show("Tuple of rationals: ", t9, separator=L",\;\;"))

t10 = ((1, 2), ("nested", (π, "deeper")))
display(l_show("Tuple nested within tuple: ", t10))

L"$\text{Tuple of rationals: } \frac{1}{2},\;\;\frac{3}{4},\;\;\frac{5}{6}$
"

L"$\text{Tuple nested within tuple: } 1,2,\text{nested},π,\text{deeper}$
"

In [18]:
t11 = (matrix=[1 2; 3 4], arraystyle=:bmatrix, color="red", fraction=3//7, text="hello", symbol=:theta, separator=L",\;\;")
display(l_show("Mixed tuple with colors: ", t11))
t11a = ((matrix=[7 2; 3 4], arraystyle=:bmatrix, color="red"), 3//7, "hello", :theta)
display(l_show("Mixed tuple with colors: ", t11a))

L"$\text{Mixed tuple with colors: } \textcolor{red}{\begin{bmatrix}
1 & 2 \\
3 & 4 \\
\end{bmatrix}},\;\;\textcolor{red}{\frac{3}{7}},\;\;\textcolor{red}{\text{hello}},\;\;\textcolor{red}{theta }$
"

L"$\text{Mixed tuple with colors: } \textcolor{red}{\begin{bmatrix}
7 & 2 \\
3 & 4 \\
\end{bmatrix}},\frac{3}{7},\text{hello},theta $
"

In [19]:
function color_offdiagonal_numbers(x, i, j, latex_str)
    # a custom formatter
    return i==j+1 || i==j-1 ? "\\textcolor{red}{$latex_str}" : latex_str
end

V=[1 2im; 3//5 im]
A=[3,5,im]
B=transpose(A')
t6a = set( "Overall set: ",
              set( "1st ", L"A =", A, arraystyle=:bmatrix, color="olive", setstyle=:array, separator=""),
              set( "2nd ", (B, color="green", arraystyle=:Varray), B', transpose(V), arraystyle=:pmatrix, separator="; ", color="red"),
              set( "3rd ", L"A^t=", A, L"\cdot", [B B A], arraystyle=:Bmatrix, color="blue", per_element_style=color_offdiagonal_numbers),
              set( "4th ", L"\frac{2}{3} B^T = ", 2//3 * B', arraystyle=:vmatrix),
    arraystyle=:array, color="black"
)

display(l_show(t6a, arraystyle=:barray, color="red", separator=L",\quad"))

"\$\\textcolor{black}{\\left\\{ \\textcolor{black}{\\text{Overall set: }} ,\\quad \\textcolor{olive}{ \\textcolor{olive}{\\text{1st }}  \\textcolor{olive}{A =}  \\textcolor{olive}{\\begin{bmatrix}\n3 \\\\\n5 \\\\\n\\mathit{i} \\\\\n\\end{bmatrix}} } ,\\quad \\textcolor{red}{\\left\\{ \\textcolor{red}{\\te" ⋯ 648 bytes ⋯ "\\mathit{i}} & \\mathit{i} \\\\\n\\end{Bmatrix}} \\right\\}} ,\\quad \\textcolor{black}{\\left\\{ \\textcolor{black}{\\text{4th }} ,\\quad \\textcolor{black}{\\frac{2}{3} B^T =} ,\\quad \\textcolor{black}{\\frac{1}{3} \\begin{vmatrix}\n6 & 10 & 2\\mathit{i} \\\\\n\\end{vmatrix}} \\right\\}} \\right\\}}\$\n"

In [20]:
l_show("Set = ", set( "A collection", (m=[1,2], color="blue", arraystyle=:barray), # need to give some name to each entry (NamedTuple)
    color="red",
    [1,2]', separator=L"\quad \rightarrow ",
    setstyle=:Barray,  arraystyle=:Vmatrix
))

L"$\text{Set = } \textcolor{red}{\left\{ \textcolor{red}{\text{A collection}} \quad \rightarrow \textcolor{blue}{\left[\begin{array}{r}
1 \\
2 \\
\end{array}\right]} \quad \rightarrow \textcolor{red}{\begin{Vmatrix}
1 & 2 \\
\end{Vmatrix}} \right\}}$
"

## 2.5. Vectors and Linear Combinations

Vectors render as column matrices.  
Use `lc(coeffs, vectors; ...)` to display linear combinations with sign control and coefficient formatting.

In [21]:
# Empty vector
display(l_show("Empty Vector: ", Vector{Int}(), arraystyle=:bmatrix))             # Should print empty [ ]

A=[1,2,3im]
# Column vector
display(l_show("Column vector: A=", A, L"\quad A^T =", transpose(A), L"\quad A^H =", A', arraystyle=:bmatrix))

# Symbolic vector: Must use arrytype Any for proper rendering of individual elements
#A =  Any[Sym("x"), Sym("y"), 3im, Sym(3)]   # Any keeps all the element types, rather than converting to a single type
#display(l_show("Symbolic Vector: ", A, L"\quad A^T =", transpose(A), L"\quad A^H =", A', arraystyle=:Bmatrix))
#print(L_show("Symbolic Vector: ", A, L"\quad A^T =", transpose(A), L"\quad A^H =", A', arraystyle=:Bmatrix))

L"$\text{Empty Vector: } \begin{bmatrix}

\end{bmatrix}$
"

L"$\text{Column vector: A=} \begin{bmatrix}
1 \\
2 \\
3\mathit{i} \\
\end{bmatrix} \quad A^T = \begin{bmatrix}
1 & 2 & 3\mathit{i} \\
\end{bmatrix} \quad A^H = \begin{bmatrix}
1 & 2 & -3\mathit{i} \\
\end{bmatrix}$
"

### :sympy backend

In [22]:
LAlatex.set_backend!(:sympy)

α = [syms("α_$i"; real=true) for i in 1:6]

s  = [ -2α[2] + 4α[5] + 7α[6],  α[2],  0,  2α[5] + 3α[6],  1,  -α[6] ]  # coefficients
X  = [ -2 -4  -2   2   4    8
        0  0  -1   1  -2   -3
       -4 -8  -3   3  10   19
        4  8   0  -1 -14  -25 ]

l_show( L"(\xi) \Leftrightarrow\;\;", lc(s, X; sign_policy=:signed, omit_one=true, drop_zero=true), "= 0")

L"$(\xi) \Leftrightarrow\;\;  -  \left(2 α_{2} + 4 α_{5} + 7 α_{6}\right)\left(\begin{array}{r}
-2 \\
0 \\
-4 \\
4 \\
\end{array}\right)  +  α_{2}\left(\begin{array}{r}
-4 \\
0 \\
-8 \\
8 \\
\end{array}\right)  +  \left(2 α_{5} + 3 α_{6}\right)\left(\begin{array}{r}
2 \\
1 \\
3 \\
-1 \\
\end{array}\right)  +  \left(\begin{array}{r}
4 \\
-2 \\
10 \\
-14 \\
\end{array}\right)  -  α_{6}\left(\begin{array}{r}
8 \\
-3 \\
19 \\
-25 \\
\end{array}\right)  = 0$
"

In [23]:
sum_clean = lc(s, X; sign_policy=:plus, omit_one=false, drop_zero=true)
l_show(sum_clean, "= 0")

L"$ \left(- 2 α_{2} + 4 α_{5} + 7 α_{6}\right)\left(\begin{array}{r}
-2 \\
0 \\
-4 \\
4 \\
\end{array}\right) + α_{2}\left(\begin{array}{r}
-4 \\
0 \\
-8 \\
8 \\
\end{array}\right) + \left(2 α_{5} + 3 α_{6}\right)\left(\begin{array}{r}
2 \\
1 \\
3 \\
-1 \\
\end{array}\right) + 1\left(\begin{array}{r}
4 \\
-2 \\
10 \\
-14 \\
\end{array}\right) + - α_{6}\left(\begin{array}{r}
8 \\
-3 \\
19 \\
-25 \\
\end{array}\right)  = 0$
"

In [24]:
sum_signed_tight = lc(s, X; sign_policy=:signed, pos=L" + ", neg=L"\;{{\color{blue}{minus}}}\;")   # default
l_show(sum_signed_tight, "= 0")

L"$ \;{{\color{blue}{minus}}}\;  \left(2 α_{2} + 4 α_{5} + 7 α_{6}\right)\left(\begin{array}{r}
-2 \\
0 \\
-4 \\
4 \\
\end{array}\right)  +  α_{2}\left(\begin{array}{r}
-4 \\
0 \\
-8 \\
8 \\
\end{array}\right)  +  \left(2 α_{5} + 3 α_{6}\right)\left(\begin{array}{r}
2 \\
1 \\
3 \\
-1 \\
\end{array}\right)  +  \left(\begin{array}{r}
4 \\
-2 \\
10 \\
-14 \\
\end{array}\right)  \;{{\color{blue}{minus}}}\;  α_{6}\left(\begin{array}{r}
8 \\
-3 \\
19 \\
-25 \\
\end{array}\right)  = 0$
"

In [25]:
lhs = lc(s, X; sign_policy=:signed, omit_one=true, drop_zero=true)
l_show(L"(\xi)\;\Leftrightarrow", lhs, "= 0")

L"$(\xi)\;\Leftrightarrow  -  \left(2 α_{2} + 4 α_{5} + 7 α_{6}\right)\left(\begin{array}{r}
-2 \\
0 \\
-4 \\
4 \\
\end{array}\right)  +  α_{2}\left(\begin{array}{r}
-4 \\
0 \\
-8 \\
8 \\
\end{array}\right)  +  \left(2 α_{5} + 3 α_{6}\right)\left(\begin{array}{r}
2 \\
1 \\
3 \\
-1 \\
\end{array}\right)  +  \left(\begin{array}{r}
4 \\
-2 \\
10 \\
-14 \\
\end{array}\right)  -  α_{6}\left(\begin{array}{r}
8 \\
-3 \\
19 \\
-25 \\
\end{array}\right)  = 0$
"

### :symbolics backend

In [26]:
set_backend!(:symbolics)

α = [syms("α_$i") for i in 1:6]
s  = [ -2α[2] - 4α[5] - 7α[6],  α[2],  0,  2α[5] + 3α[6],  1,  -α[6] ]  # coefficients
X  = [ -2 -4  -2   2   4    8
        0  0  -1   1  -2   -3
       -4 -8  -3   3  10   19
        4  8   0  -1 -14  -25 ]

l_show( L"(\xi) \Leftrightarrow\;\;", lc(s, X; sign_policy=:signed, omit_one=true, drop_zero=true), "= 0")

L"$(\xi) \Leftrightarrow\;\;  \left(- 2 ~ \alpha_2 - 4 ~ \alpha_5 - 7 ~ \alpha_6\right)\left(\begin{array}{r}
-2 \\
0 \\
-4 \\
4 \\
\end{array}\right)  +  \alpha_2\left(\begin{array}{r}
-4 \\
0 \\
-8 \\
8 \\
\end{array}\right)  +  \left(2 ~ \alpha_5 + 3 ~ \alpha_6\right)\left(\begin{array}{r}
2 \\
1 \\
3 \\
-1 \\
\end{array}\right)  +  \left(\begin{array}{r}
4 \\
-2 \\
10 \\
-14 \\
\end{array}\right)  +  \left(- \alpha_6\right)\left(\begin{array}{r}
8 \\
-3 \\
19 \\
-25 \\
\end{array}\right)  = 0$
"

## 2.6. Standard Matrices

Matrix literals, transposes, and adjoints all work. Pick the LaTeX environment with `arraystyle` (e.g., `:pmatrix`, `:bmatrix`).


In [27]:
A = [1 2; 3 4]
display(l_show("integer matrix A=", A))  # Expected: LaTeX pmatrix of [[1,2]; [3,4]]

B = [1.5 2.5; 3.5 4.5]
display(l_show("real matrix B=", B, arraystyle=:bmatrix))  # Expected: LaTeX bmatrix

C = [1//2 1//3; 1//4 1//5]
display(l_show("not factored C=", C, factor_out=false))     # Expected: Fraction elements in LaTeX
display(l_show("default: factored C=", C, per_element_style=tril_formatter))   # Expected: Fraction factored out

# Colored and formatted matrices
display(l_show("green with braces A=", (A, color="green", arraystyle=:Bmatrix)))  # Green curly braces matrix
display(l_show("bold C=", (C, per_element_style=bold_formatter)))  # Bold entries

# number formatting
display(l_show(L"C \approx", (C, factor_out=false, number_formatter=x->round_value(x,2))))


L"$\text{integer matrix A=} \left(\begin{array}{rr}
1 & 2 \\
3 & 4 \\
\end{array}\right)$
"

L"$\text{real matrix B=} \begin{bmatrix}
1.5 & 2.5 \\
3.5 & 4.5 \\
\end{bmatrix}$
"

L"$\text{not factored C=} \left(\begin{array}{rr}
\frac{1}{2} & \frac{1}{3} \\
\frac{1}{4} & \frac{1}{5} \\
\end{array}\right)$
"

L"$\text{default: factored C=} \frac{1}{60} \left(\begin{array}{rr}
\textcolor{red}{30} & 20 \\
\textcolor{red}{15} & \textcolor{red}{12} \\
\end{array}\right)$
"

L"$\text{green with braces A=} \textcolor{green}{\begin{Bmatrix}
1 & 2 \\
3 & 4 \\
\end{Bmatrix}}$
"

L"$\text{bold C=} \frac{1}{60} \left(\begin{array}{rr}
\boldsymbol{30} & \boldsymbol{20} \\
\boldsymbol{15} & \boldsymbol{12} \\
\end{array}\right)$
"

L"$C \approx \left(\begin{array}{rr}
0.5 & 0.33 \\
0.25 & 0.2 \\
\end{array}\right)$
"

In [28]:
A = [1 2 0 0;
     3 4 0 0;
     0 0 5 6;
     0 0 7 8]

l_show("A = ",
    (a=A,
     per_element_style=(x,i,j,fx) ->
         diagonal_blocks_formatter(x,i,j,fx; blocks=[2,-1,1])),  # will all be red by default, second block skipped
    number_formatter = x -> round(x, digits = 1)
)

L"$\text{A = } \left(\begin{array}{rrrr}
\textcolor{red}{1.0} & \textcolor{red}{2.0} & 0.0 & 0.0 \\
\textcolor{red}{3.0} & \textcolor{red}{4.0} & 0.0 & 0.0 \\
0.0 & 0.0 & 5.0 & 6.0 \\
0.0 & 0.0 & 7.0 & \textcolor{red}{8.0} \\
\end{array}\right)$
"

## 2.7. Complex & Symbolic Matrices

Symbolics and SymPy entries can mix with complex rationals.  
If you hit promotion errors, build arrays with `mixed_matrix` or `@mixed_matrix`.


In [29]:
D = [1 2im//3; (-1)//2 5//2im]
display(l_show("Complex Matrix D =", D, L"\quad D^T =", transpose(D), L"\qquad D^H =", D'))  # Expected: Matrix with complex numbers

set_backend!(:symbolics)
x, y = LAlatex.syms(:x, :y)
E = [x 2; 3 y]
display(l_show("Matrix with symbols ", E))  # Expected: Matrix with symbols x and y

# Complex + Symbols: macro works for both backends
F = @mixed_matrix([1//2 x; (1+im)//3 2*y])
display(l_show("Complex matrix with symbols ", F, color="purple"))

L"$\text{Complex Matrix D =} \frac{1}{6} \left(\begin{array}{rr}
6 & 4\mathit{i} \\
-3 & -15\mathit{i} \\
\end{array}\right) \quad D^T = \frac{1}{6} \left(\begin{array}{rr}
6 & -3 \\
4\mathit{i} & -15\mathit{i} \\
\end{array}\right) \qquad D^H = \frac{1}{6} \left(\begin{array}{rr}
6 & -3 \\
-4\mathit{i} & 15\mathit{i} \\
\end{array}\right)$
"

L"$\text{Matrix with symbols } \left(\begin{array}{rr}
x & 2 \\
3 & y \\
\end{array}\right)$
"

L"$\textcolor{purple}{\text{Complex matrix with symbols }} \textcolor{purple}{\frac{1}{6} \left(\begin{array}{rr}
3 & 6 ~ x \\
2+2\mathit{i} & 12 ~ y \\
\end{array}\right)}$
"

## 2.8. Special Matrices in Julia

Diagonal, sparse, and other structured matrices are converted to dense display for LaTeX output.


In [30]:
using LinearAlgebra, SparseArrays, BlockArrays

# Adjoint (Transpose) Matrix
A_adj = adjoint(A)
display(l_show("Transpose: ", A_adj))  # Expected: Transposed bmatrix

# Symmetric and Hermitian Matrices
S = Symmetric(A)
display(l_show("Symmetric ", S))  # Expected: Symmetric matrix

H = Hermitian(A)
display(l_show("Hermitian ", H))  # Expected: Hermitian matrix

# Diagonal Matrix
D = Diagonal([1,2,3])
display(l_show("Diagonal ", D))  # Expected: Diagonal matrix

# Sparse Matrix
S = sparse([1,2], [1,2], [10,20])
display(l_show("Sparse ", S))  # Expected: Sparse matrix converted to full matrix


L"$\text{Transpose: } \left(\begin{array}{rrrr}
1 & 3 & 0 & 0 \\
2 & 4 & 0 & 0 \\
0 & 0 & 5 & 7 \\
0 & 0 & 6 & 8 \\
\end{array}\right)$
"

L"$\text{Symmetric } \left(\begin{array}{rrrr}
1 & 2 & 0 & 0 \\
2 & 4 & 0 & 0 \\
0 & 0 & 5 & 6 \\
0 & 0 & 6 & 8 \\
\end{array}\right)$
"

L"$\text{Hermitian } \left(\begin{array}{rrrr}
1 & 2 & 0 & 0 \\
2 & 4 & 0 & 0 \\
0 & 0 & 5 & 6 \\
0 & 0 & 6 & 8 \\
\end{array}\right)$
"

L"$\text{Diagonal } \left(\begin{array}{rrr}
1 & 0 & 0 \\
0 & 2 & 0 \\
0 & 0 & 3 \\
\end{array}\right)$
"

L"$\text{Sparse } \left(\begin{array}{rr}
10 & 0 \\
0 & 20 \\
\end{array}\right)$
"

## 2.9. BlockArrays

BlockArrays keep visual separators between blocks, so you can show block structure directly in LaTeX.


In [31]:
A = BlockArray(
    [1 2 3; 4 5 6; 7 8 9],  # Regular matrix
    [1, 2],  # Row blocks: [1, 2]
    [1, 2]   # Column blocks: [1, 2]
)
l_show( "block array of integers A=", A, arraystyle=:bmatrix)

L"$\text{block array of integers A=} \left[\begin{array}{r|rr}
1 & 2 & 3 \\ \hline
4 & 5 & 6 \\
7 & 8 & 9 \\
\end{array}\right]$
"

In [32]:
C = BlockArray(
    fill(1.0, 4, 6),  # Regular matrix
    [2, 2],  # Row blocks: [2, 2]
    [3, 3]   # Column blocks: [3, 3]
)
l_show("block array of float, C=", C, number_formatter=x->round_value(x,0), arraystyle=:vmatrix)

L"$\text{block array of float, C=} \left|\begin{array}{rrr|rrr}
1 & 1 & 1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1 & 1 & 1 \\ \hline
1 & 1 & 1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1 & 1 & 1 \\
\end{array}\right|$
"

In [33]:
D = BlockArray(
    [1+im  2-im  3+2im;
     4-2im  5+im  6-im;
     7+im  8-im  9+2im],  # Complex integer matrix
    [1, 2],               # Row blocks: [1, 2]
    [1, 1, 1]             # Column blocks
)
l_show("blockarray of complex ints: D=", D)


L"$\text{blockarray of complex ints: D=} \left(\begin{array}{r|r|r}
1+\mathit{i} & 2-\mathit{i} & 3+2\mathit{i} \\ \hline
4-2\mathit{i} & 5+\mathit{i} & 6-\mathit{i} \\
7+\mathit{i} & 8-\mathit{i} & 9+2\mathit{i} \\
\end{array}\right)$
"

In [34]:
E = BlockArray(
    [(1+1im//3)  ((2-1im)//2)  3+2im;
     4-2im  5//2+im  6-im//10;
     7+im//4  8-im  9+2im],  # Complex rational matrix
    [1, 2],  # Row blocks
    [1, 2]   # Column blocks
)
highlight = (x, i, j, formatted) -> i ==1 && j == 1 ? "\\boldsymbol{\\textcolor{red}{{$formatted}}}" : "\\boldsymbol{{$formatted}}"
l_show(E, per_element_style=highlight)

L"$\frac{1}{60} \left(\begin{array}{r|rr}
\boldsymbol{\textcolor{red}{{60+20\mathit{i}}}} & \boldsymbol{{60-30\mathit{i}}} & \boldsymbol{{180+120\mathit{i}}} \\ \hline
\boldsymbol{{240-120\mathit{i}}} & \boldsymbol{{150+60\mathit{i}}} & \boldsymbol{{360-6\mathit{i}}} \\
\boldsymbol{{420+15\mathit{i}}} & \boldsymbol{{480-60\mathit{i}}} & \boldsymbol{{540+120\mathit{i}}} \\
\end{array}\right)$
"

In [35]:
A = BlockArray(
    [1 2 4; 3 4 5],  # Regular matrix
    [1, 1],      # Row blocks
    [2, 1]       # Column blocks
)

function color_odd_numbers(x, i, j, latex_str)
    return isodd(x) ? "\\textcolor{red}{$latex_str}" : latex_str
end

# Test L_show with per_element_style on BlockArray
display(l_show("Colorize odd values,  A=", A; per_element_style=color_odd_numbers))

L"$\text{Colorize odd values,  A=} \left(\begin{array}{rr|r}
\textcolor{red}{1} & 2 & 4 \\ \hline
\textcolor{red}{3} & 4 & \textcolor{red}{5} \\
\end{array}\right)$
"

## 2.10. Per Element Formatting

`per_element_style` receives `(x, i, j, latex_str)` so you can style cells based on position or value.


In [36]:
CC=copy(C)
for i in 1:4 CC[i,i]=2. end

diagonal_highlight = (x, i, j, formatted) -> i == j ? "\\textcolor{red}{" * formatted * "}" : formatted
l_show((CC, per_element_style=diagonal_highlight))

L"$\left(\begin{array}{rrr|rrr}
\textcolor{red}{2.0} & 1.0 & 1.0 & 1.0 & 1.0 & 1.0 \\
1.0 & \textcolor{red}{2.0} & 1.0 & 1.0 & 1.0 & 1.0 \\ \hline
1.0 & 1.0 & \textcolor{red}{2.0} & 1.0 & 1.0 & 1.0 \\
1.0 & 1.0 & 1.0 & \textcolor{red}{2.0} & 1.0 & 1.0 \\
\end{array}\right)$
"

In [37]:
row_color = (x, i, j, formatted) -> i % 2 == 0 ? "\\boxed{\\textcolor{purple}{" * formatted * "}}" : formatted
P=[1 2 3 4 5; 6 7 8 9 10; -1 -2 -3 -4 -5; -6 -7 -8 -9 -10]
l_show(( P, per_element_style=row_color, arraystyle=:array))

L"$\begin{array}{rrrrr}
1 & 2 & 3 & 4 & 5 \\
\boxed{\textcolor{purple}{6}} & \boxed{\textcolor{purple}{7}} & \boxed{\textcolor{purple}{8}} & \boxed{\textcolor{purple}{9}} & \boxed{\textcolor{purple}{10}} \\
-1 & -2 & -3 & -4 & -5 \\
\boxed{\textcolor{purple}{-6}} & \boxed{\textcolor{purple}{-7}} & \boxed{\textcolor{purple}{-8}} & \boxed{\textcolor{purple}{-9}} & \boxed{\textcolor{purple}{-10}} \\
\end{array}$
"

## 2.11. Combining Different Types

You can mix text, arrays, sets, tuples, and symbols in one `l_show` call to build a full expression in a single display.  
Argument parsing may require passing namedd tuples, e.g.,  
$\qquad$ (b=B, color="red", arraystyle=:bmatrix) $\;$ rather than  
$\qquad$ (B, color="red", arraystyle=:bmatrix)


In [38]:
display(l_show(
    "Equation:",
    (A, color="blue", arraystyle=:pmatrix),
    L"\ne",
    (B, color="red", arraystyle=:bmatrix),
    L";\qquad x =",
    (; value=1//3, color="green"),  # need a named tuple to apply options to a scalar
    L"\quad", "i.e., ", 1//3,
    color="purple"                  # global option
))

L"$\textcolor{purple}{\text{Equation:}} \textcolor{blue}{\left(\begin{array}{rr|r}
1 & 2 & 4 \\ \hline
3 & 4 & 5 \\
\end{array}\right)} \textcolor{purple}{\ne} \textcolor{red}{\begin{bmatrix}
1.5 & 2.5 \\
3.5 & 4.5 \\
\end{bmatrix}} \textcolor{purple}{;\qquad x =} \textcolor{green}{\frac{1}{3}} \textcolor{purple}{\quad} \textcolor{purple}{\text{i.e., }} \textcolor{purple}{\frac{1}{3}}$
"

## 2.12. Inline vs Block LaTeX

Use `inline=true` for `$...$` or `inline=false` for `\[...\]` blocks.


In [39]:
display(l_show("left justified: A = ", A, inline=true))   # Inline: Expected `$ \begin{bmatrix} ... \end{bmatrix} $`
display(l_show("centered: A = ", A, inline=false))  # Block: Expected `\[ \begin{bmatrix} ... \end{bmatrix} \]`

L"$\text{left justified: A = } \left(\begin{array}{rr|r}
1 & 2 & 4 \\ \hline
3 & 4 & 5 \\
\end{array}\right)$
"

L"\[\text{centered: A = } \left(\begin{array}{rr|r}
1 & 2 & 4 \\ \hline
3 & 4 & 5 \\
\end{array}\right)\]
"

## 2.13. Obtaining Raw LaTeX Representations

`L_show` returns raw LaTeX (with math delimiters) so you can paste into documents or reuse in generated output.


In [40]:
# print the output of L_show (capitalized) instead of l_show

println(L_show(
    "Equation:",
    (A, color="blue", arraystyle=:pmatrix),
    L"\ne",
    (B, color="red", arraystyle=:array),
    L";\qquad x =",
    (; value=1//3, color="green"),  # need a named tuple to apply options to a scalar
    L"\quad", "i.e., ", 1//3,
    color="purple"                  # global option
))

$\textcolor{purple}{\text{Equation:}} \textcolor{blue}{\left(\begin{array}{rr|r}
1 & 2 & 4 \\ \hline
3 & 4 & 5 \\
\end{array}\right)} \textcolor{purple}{\ne} \textcolor{red}{\begin{array}{rr}
1.5 & 2.5 \\
3.5 & 4.5 \\
\end{array}} \textcolor{purple}{;\qquad x =} \textcolor{green}{\frac{1}{3}} \textcolor{purple}{\quad} \textcolor{purple}{\text{i.e., }} \textcolor{purple}{\frac{1}{3}}$

